# Lesson 11 Lab — Tensor, Pipeline, Data, and Expert Parallelism

**Puzzle:** How should a 70B service map onto eight GPUs when one RTX 5090 cannot reproduce that topology?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Parallelism is a placement decision constrained by model size, communication, request isolation, and cluster topology. Choosing `tensor_parallel_size=8` because eight devices exist can place collective traffic across a slow boundary and reduce useful throughput.


## 0. Predict before running

1. Eliminate layouts that cannot fit 70B BF16 weights.
2. Compare estimated cross-node bytes for TP8 and TP4×DP2.
3. Name the NCCL trace required before a native claim.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab records the real single-GPU environment, reads the installed vLLM parallel CLI surface, and evaluates a transparent eight-GPU placement model for TP, PP, DP, and hybrid layouts across two four-GPU nodes.

- Model fit is a hard constraint before throughput optimization.
- TP communication happens inside the model step and is topology-sensitive.
- DP increases replica concurrency only when each replica can fit the model.


## 2. Derive the mechanism

Tensor parallelism shards layer operations and communicates on many layers. Pipeline parallelism assigns layer stages and introduces bubbles or microbatch scheduling. Data parallel replicas own separate request batches and normally duplicate weights. Expert parallelism shards routed experts while preserving dense/shared components. Communication frequency and link bandwidth must be matched to the topology.

### Mechanism at a glance

```mermaid
flowchart TD
  M["model + KV memory"] --> F{"fits one GPU?"}
  F -->|"yes"| D["data-parallel replicas"]
  F -->|"no"| T["tensor or pipeline shards"]
  T --> N{"fast links within node?"}
  N -->|"yes"| H["TP inside node + DP across nodes"]
  N -->|"no"| P["revisit PP / quantization / capacity"]
```

### Walk it step by step

1. **Solve memory fit.** Remove layouts that cannot hold weights, KV, and headroom.
2. **Map communication.** Mark which collectives cross NVLink, PCIe, or the network.
3. **Choose replication.** Use DP only after a complete replica fits.
4. **Prove natively.** Collect per-rank traces and throughput on the actual topology.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 11
LESSON_TITLE = 'Tensor, Pipeline, Data, and Expert Parallelism'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260823
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | TP8 spanning two nodes |
| Candidate | TP4 within each node plus DP2, and PP alternatives |
| Held constant | eight-GPU topology, model bytes, per-GPU memory, link assumptions, and batch |
| Measurements | fit, replica count, cross-node communication estimate, and exposed CLI flags |
| Evidence | `capacity-model` |

**Experiment:** Evaluate candidate placements with explicit weight, KV, link, and collective assumptions; probe available engine arguments.


## 5. Inspect the experiment code

Every formula and assumed bandwidth is emitted in the artifact. The experiment does not initialize distributed processes, so all multi-GPU performance rows remain planning estimates.

Do not execute until the code matches the frozen table.


In [2]:
parameter_billion=70.0; weight_gib=parameter_billion*1e9*2/2**30
gpu_gib=torch.cuda.get_device_properties(0).total_memory/2**30; reserve_gib=5.0; activation_gib=3.0
_,serve_help=cli_help("serve")
def layout(tp,dp,pp,cross_fraction):
    shard=weight_gib/(tp*pp)
    return {"tp":tp,"dp":dp,"pp":pp,"replicas":dp,"weight_gib_per_gpu":shard,
            "fits":shard+reserve_gib<gpu_gib,"cross_node_gib_step":activation_gib*cross_fraction}
metrics={"topology":{"nodes":2,"gpus":8,"gpus_per_node":4,"gpu_gib":gpu_gib},
         "assumptions":{"model_parameters_billion":parameter_billion,"bf16_weight_gib":weight_gib,
                        "reserve_gib_per_gpu":reserve_gib,"activation_gib_step":activation_gib},
         "layouts":{"tp8":layout(8,1,1,.5),"tp4_dp2":layout(4,2,1,0),"tp4_pp2":layout(4,1,2,.25)},
         "cli":{"tensor_parallel":"--tensor-parallel-size" in serve_help,
                "pipeline_parallel":"--pipeline-parallel-size" in serve_help,
                "data_parallel":"--data-parallel-size" in serve_help},"native_multi_gpu_executed":False}
analysis=(f"The ledger estimates {weight_gib:.1f} GiB BF16 weights. TP8/TP4×DP2 fit="
          f"{metrics['layouts']['tp8']['fits']}/{metrics['layouts']['tp4_dp2']['fits']}; only the "
          "modeled TP8 collective crosses nodes. No distributed run occurred.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| GPU count | 8 |
| GPUs per node | 4 |
| TP8 fits | yes |
| TP4-DP2 fits | no |
| TP8 cross-node bytes | 1.500000 |
| TP4-DP2 replicas | 2 |
| TP flag available | no |


## 7. Explain the result

The ledger estimates 130.4 GiB BF16 weights. TP8/TP4×DP2 fit=True/False; only the modeled TP8 collective crosses nodes. No distributed run occurred.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured environment facts feed explicit planning arithmetic. Assumed topology, demand, bandwidth, and reserve fields remain assumptions until a native deployment test.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 11, "title": 'Tensor, Pipeline, Data, and Expert Parallelism', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The capacity model rejects impossible or topology-hostile layouts; it does not measure multi-GPU vLLM performance.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 11,
  "title": "Tensor, Pipeline, Data, and Expert Parallelism",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260823
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "topology": {
      "nodes": 2,
      "gpus": 8,
      "gpus_per_node": 4,
      "gpu_gib": 31.35833740234375
    },
    "assumptions": {
      "model_parameters_billion": 70.0,
      "bf16_weight_gib": 130.385160446167,
      "reserve_gib_per_gpu": 5.0,
      "activation_gib_step": 3.0
    },
    "layouts": {
      "tp8": {
        "tp": 8,
        "dp": 1,
        "pp": 1,
        "replicas": 1,
        "weight_gib_per_gpu": 16.298145055770874,
        "fits": true,
        "cross_node_gib_step": 1.5
      },
      "tp4_dp2": {
        "tp": 4,
        "dp": 2,
        "pp": 1,
        "repli

## 9. Make the bounded decision

> The capacity model rejects impossible or topology-hostile layouts; it does not measure multi-GPU vLLM performance.

**Acceptance/rollback:** Select only layouts that fit with headroom, keep frequent collectives on fast links, and then pass a native multi-node benchmark.

**Failure analysis:** Collective algorithms, overlap, quantized weights, expert routing, uneven layers, and scheduler behavior can dominate the simplified estimate. One RTX 5090 cannot validate NCCL topology.


## 10. Extend the evidence

Run the selected two-node layout with NCCL traces, per-rank memory, failure injection, and identical request replay; compare against the best single-node baseline.

The full boundary and references are in [`README.md`](README.md).
